In [ ]:
import scipy.io as sio
import matplotlib.pyplot as plt
import numpy as np
import torch
import os
import warnings
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras import optimizers
from scipy import signal
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error
warnings.filterwarnings('ignore')

In [ ]:
datapath = '../input/BloodPressureDataset/'

In [ ]:
def load_data_partial(filename):
  mat_contents = sio.loadmat(filename) 

  return mat_contents

In [ ]:
def load_data(fileDir, exercise):
    word = exercise.lower()
    file_path_list = []
    valid_file_extensions = [".mat"]
    valid_file_extensions = [item.lower() for item in valid_file_extensions]


    for file in os.listdir(fileDir):
        extension = os.path.splitext(file)[1]
        if extension.lower() not in valid_file_extensions:
            continue
        file_path_list.append(os.path.join(fileDir, file))

    Data = []
    for path in file_path_list:
        base=os.path.basename(path)
        base = os.path.splitext(base)[0]
        if word in base:
            print(fileDir+'/%s'%(base))
            mat_contents = sio.loadmat(fileDir+'/%s'%(base))
            val = mat_contents['p']
            total_array = val[0,:] #assigning an array
            Data.append(total_array)

    return Data

Loading the MAT files

In [ ]:
word = 'part'
total_data = load_data(datapath, word)
x =load_data_partial(datapath+str('part_1.mat'))

In [ ]:
x.keys()


In [ ]:
abp_list = x['p'][0,1]
print(abp_list.shape)
abp = x['p'][0,1][1,:1000]


x_max = max(abp_list[1,:])
x_min = min(abp_list[1,:])


plt.figure()
plt.plot(abp)

nrm_abp = (abp-x_min)/(x_max-x_min)  # Normalizing Data between 0 and 1
plt.figure()
plt.plot(nrm_abp)


ret_abp = nrm_abp*(x_max - x_min) +x_min   #Reversed from normalized value
plt.figure()
plt.plot(ret_abp)

Setting the configuration parameters for the models

In [ ]:
config = {
    'seed': 77,
    'sample_size': 1000,
    'train_size': 3000000, # train on 3 million data points
    'train_params':{
        'batch_size':32,
        'epochs': 50,
        'learn_rate': 0.0001,#1e-3,
    }
}

In [ ]:
# Initializing TPU
tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
tf.config.experimental_connect_to_cluster(tpu)
tf.tpu.experimental.initialize_tpu_system(tpu)

# Instantiate distribution strategy
tpu_strategy = tf.distribute.experimental.TPUStrategy(tpu)
    
auto = tf.data.experimental.AUTOTUNE
replicas = tpu_strategy.num_replicas_in_sync
Batch_size = config['train_params']['batch_size'] * replicas
print(f'Number of Replicas: {replicas}')

In [ ]:
def SetSeed(seed=None):
    """Set seed for reproducibility."""
    tf.random.set_seed(seed)
    np.random.seed(seed)
    
SetSeed(seed=config['seed'])

In [ ]:
print(len(total_data))
print(len(total_data[1]))
print(total_data[1][1].shape)

In [ ]:
print(total_data[0][1])

Feature Extraction and Feature Scaling: Extracting PPG ,ECG and ABP values

In [ ]:
PPG = []
ABP = []
ECG = []
i=0
for j in range(len(total_data[i])):
  k = len(total_data[i][j][0,:])
  for n in range(k//1000):
      ppg = (total_data[i][j][0,(n*1000):(n*1000)+1000])
      abp = (total_data[i][j][1,(n*1000):(n*1000)+1000])
      ecg = (total_data[i][j][2,(n*1000):(n*1000)+1000])
      PPG.append(ppg)
      ABP.append(abp)
      ECG.append(ecg)


PPG = np.asarray(PPG)
ABP = np.asarray(ABP)
ECG = np.asarray(ECG)


In [ ]:
PPG

In [ ]:
PPG.shape

In [ ]:
len(PPG)

In [ ]:
# ##plotting sample  ppg, ecg and bp signals
# ##using a sample size of 125
fig, ax = plt.subplots(3,1, figsize=(9,12), sharex=True)
y = 1000
ax[0].set_title('PPG graph', fontsize=16)
ax[0].set_ylabel('Signal Value')
ax[0].plot(PPG[y,:], c = 'dodgerblue')

ax[1].set_title('ECG graph', fontsize=16)
ax[1].set_ylabel('Signal Value')
ax[1].plot(ECG[y,:], c='darkorange')

ax[2].set_title('Blood Pressure (BP) graph', fontsize=16)
ax[2].set_ylabel('Signal Value')
ax[2].set_xlabel('Sample size')
ax[2].plot(ABP[y,:], c = 'red')

In [ ]:
# creating train and val sets
X_train1, X_test1, y_train1, y_test1 = train_test_split(PPG, ABP, test_size=0.30)

In [ ]:
X_train1

In [ ]:
y_train1

Linear Model without Normalizing PPG,ECG and ABP values

In [ ]:
#Linear Regression with Values of PPG
import numpy as np
from sklearn.linear_model import LinearRegression
linear_model_std = LinearRegression()
linear_model_std.fit(X_train1, y_train1)

In [ ]:
y_pred1 = linear_model_std.predict(X_test1)

In [ ]:
rmse= np.sqrt(np.mean((y_pred1-y_test1)**2))
print(f'LR standard model RMSE is: {rmse}')

In [ ]:
#Checking MAE error for LR model (not normalized input and output)
LR_std__MAE= tf.keras.metrics.MeanAbsoluteError()
LR_std__MAE.update_state(y_test1, y_pred1)
print(f'LR_std_Model MAE: {LR_std__MAE.result().numpy()}')

In [ ]:
# Checking if y_test1 and y_pred1 have any NaN values
print(len(y_test1))
print(len(y_pred1))
print(np.isnan(y_test1).any())
print(np.isnan(y_pred1).any())
print(np.argwhere(np.isnan(y_test1)))

In [ ]:
#Checking MAP for LR model
score = []
for i in range(len(y_test1)):
  score.append(abs(y_test1[i]-y_pred1[i]))

print("Average MAP error for LR standard model is: %f mmHG" %(np.mean(score)))

In [ ]:
# Visualize predicted BP and the True BP in standard Linear Model
plt.title("===True BP values Vs Predicted BP values in standard LR Model===")
plt.xlabel('Number of samples taken')
plt.ylabel('BP values')
plt.plot(y_test1[1000,:]) #only plotting 100 samples
plt.plot(y_pred1[1000,:])
plt.legend(['True_BP', 'Predicted_BP'])

Normalization

In [ ]:
def normalise(x):
  normalised = (x-min(x))/(max(x)-min(x))
  return normalised

def scale_abp(x):
  normalised = x/200
  return normalised

def normalise_abp(abp, x_max, x_min):
  normalised = (abp-x_min)/(x_max-x_min)
  return normalised

def abp_maxmin_value(x):
  max_x = []
  min_x = []
  for i in range(len(x)):
    for j in range(len(x[i])):
      max_x.append(max(x[i][j][1,:]))
      min_x.append(min(x[i][j][1,:]))
  x_max = max(max_x)
  x_min = min(min_x)

  return x_max, x_min

In [ ]:
x_max, x_min = abp_maxmin_value(total_data)
print(x_max, x_min)
x_max = 200.0
x_min = 50.0

Feature Extraction and Feature Scaling: Extracting PPG ,ECG and ABP values and normalizing them

In [ ]:
# #Data Pre-processing
# # Extracting signal data Feature-extraction
PPG_N = []
ABP_N = []
ECG_N = []
i=0
for j in range(len(total_data[i])):
  k = len(total_data[i][j][0,:])
  for n in range(k//1000):
      ppg_n = normalise(total_data[i][j][0,(n*1000):(n*1000)+1000])
      abp_n = normalise_abp(total_data[i][j][1,(n*1000):(n*1000)+1000], x_max, x_min)
      ecg_n = normalise(total_data[i][j][2,(n*1000):(n*1000)+1000])
      PPG_N.append(ppg_n)
      ABP_N.append(abp_n)
      ECG_N.append(ecg_n)


PPG_N = np.asarray(PPG_N)
ABP_N = np.asarray(ABP_N)
ECG_N = np.asarray(ECG_N)


In [ ]:
PPG_N

In [ ]:
PPG_N.shape

In [ ]:
len(PPG_N)

In [ ]:
# ##plotting sample normalized ppg, ecg and bp signals
# ##using a sample size of 125
fig, ax = plt.subplots(3,1, figsize=(9,12), sharex=True)
y = 1000
ax[0].set_title('Normalized PPG graph', fontsize=16)
ax[0].set_ylabel('Signal Value')
ax[0].plot(PPG_N[y,:], c = 'dodgerblue')

ax[1].set_title('Normalized ECG graph', fontsize=16)
ax[1].set_ylabel('Signal Value')
ax[1].plot(ECG_N[y,:], c='darkorange')

ax[2].set_title('Normalized Blood Pressure (ABP) graph', fontsize=16)
ax[2].set_ylabel('Signal Value')
ax[2].set_xlabel('Sample size')
ax[2].plot(ABP_N[y,:], c = 'red')

In [ ]:
# creating train and val sets
X_train, X_test, y_train, y_test = train_test_split(PPG_N, ABP_N, test_size=0.30)

In [ ]:
X_train

In [ ]:
X_train.shape

In [ ]:
y_train

Linear Model After Normalizing PPG,ECG and ABP values

In [ ]:
#Linear Regression with Normalized Values of PPG
import numpy as np
from sklearn.linear_model import LinearRegression
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

In [ ]:
y_pred = linear_model.predict(X_test)

In [ ]:
rmse= np.sqrt(np.mean((y_pred-y_test)**2))
print(f'LR model RMSE is: {rmse}')

In [ ]:
#Checking MAE error for Linear Regression Model
LR_MAE= tf.keras.metrics.MeanAbsoluteError()
LR_MAE.update_state(y_test, y_pred)
print(f'LR Model MAE: {LR_MAE.result().numpy()}')

In [ ]:
# Checking if y_test and y_pred have any NaN values
print(len(y_test))
print(len(y_pred))
print(np.isnan(y_test).any())
print(np.isnan(y_pred).any())
print(np.argwhere(np.isnan(y_test)))

In [ ]:
#Checking MAP for LR model
score = []
for i in range(len(y_test)):
  score.append(abs(y_test[i]-y_pred[i]))

print("Average MAP error for LR model is: %f mmHG" %(np.mean(score)))

In [ ]:
# Visualize predicted BP and the True BP in standard LR model
plt.title("===True BP values Vs Predicted BP values in LR Model(Normalized)===")
plt.xlabel('Number of samples taken')
plt.ylabel('BP values')
plt.plot(y_test[1000,:]) #only plotting 100 samples
plt.plot(y_pred[1000,:])
plt.legend(['True_BP', 'Predicted_BP'])

Artificial Neural Network Model

In [ ]:
# Baseline Sequential Model
from tensorflow.keras import backend as K
K.clear_session() 

def Model(input_dim, num_class):
    model = Sequential()

    model.add(layers.Dense(2048, input_dim = input_dim, activation='sigmoid'))
    # model.add(layers.LSTM(512, input_dim = input_dim, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1024, 'relu'))
    model.add(layers.Dropout(0.25))
    model.add(layers.Dense(512, 'relu')) 
    model.add(layers.Dropout(0.25))
    # model.add(layers.Dense(64, 'relu'))    
    # model.add(layers.Dropout(0.25))
    model.add(layers.Dense(num_class, 'linear'))
    
    
    return model

Input_dim = X_train.shape[1]
Classes = 1000

# Creating the model on TPU
with tpu_strategy.scope():
  model = Model(input_dim=Input_dim, num_class=Classes)
  model.compile(loss=tf.keras.losses.MeanAbsoluteError(),
                  optimizer='adam',
                  metrics=[tf.keras.metrics.RootMeanSquaredError()]
                 )
  model.summary()

In [ ]:
# Training the ANN model
ANN_model_train = model.fit(X_train[:config['train_size']], # using the first 3 million rows.
                    y_train[:config['train_size']],
                    validation_split=0.1,
                    epochs=config['train_params']['epochs'],
                    batch_size=Batch_size,
                    verbose = 1
                   )

In [ ]:
#Predicting on the test set using the ANN( artificial neural network) model
ANN_predictions = model.predict(X_test[:config['train_size']])
rmse = tf.keras.metrics.RootMeanSquaredError()
rmse.update_state(y_test[:config['train_size']], ANN_predictions)
print(f' ANN Model RMSE: {rmse.result().numpy()}')

In [ ]:
len(ANN_predictions)

In [ ]:
type(ANN_predictions)

In [ ]:
ANN_predictions

In [ ]:
#Checking MAE error for ANN Model
ANN_MAE= tf.keras.metrics.MeanAbsoluteError()
ANN_MAE.update_state(y_test, ANN_predictions)
print(f'ANN Model MAE: {ANN_MAE.result().numpy()}')

In [ ]:
# Checking if y_test and ANN_predictions have any NaN values
print(len(y_test))
print(len(ANN_predictions))
print(np.isnan(y_test).any())
print(np.isnan(ANN_predictions).any())
print(np.argwhere(np.isnan(y_test)))

In [ ]:
#Checking MAP error for ANN model
score = []
for i in range(len(y_test)):
  score.append(abs(y_test[i]-ANN_predictions[i]))

print("Average MAP error for ANN model is: %f mmHG" %(np.mean(score)))

In [ ]:
#Visualizing ANN model Train Loss vs RMSE
plt.title('Train loss against Root_Mean_Squared_error in ANN model')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.plot(ANN_model_train.history['loss'])
plt.plot(ANN_model_train.history['root_mean_squared_error'])
plt.legend(['Loss', 'Root_Mean_Squared_error'])

In [ ]:
# Visualize predicted BP and the True BP in ANN model
plt.title("===True BP values Vs Predicted BP values in ANN Model===")
plt.xlabel('Number of samples taken')
plt.ylabel('BP values')
plt.plot(y_test[1000,:]) #only plotting 100 samples
plt.plot(ANN_predictions[1000,:])
plt.legend(['True_BP', 'Predicted_BP'])

Reshaping the X_train and X_test values for LSTM model

In [ ]:
X_train_reshape = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

In [ ]:
X_train_reshape.shape

In [ ]:
X_train_reshape

In [ ]:
X_test_reshape =np.reshape(X_test,(X_test.shape[0] ,X_test.shape[1],1))

In [ ]:
X_test_reshape.shape

In [ ]:
X_test_reshape

LSTM Model

In [ ]:
from tensorflow.keras import backend as K
from tensorflow.keras import Model as md
from tensorflow.keras import regularizers
K.clear_session()
def LSTM_Model(input_dim,num_class):
  model = Sequential()
  model.add(layers.LSTM(64, return_sequences=True, input_shape = (X_train_reshape.shape[1],1)))
  #model.add(layers.Dense(25,activation='relu'))
  model.add(layers.Dropout(0.5))
  model.add(layers.LSTM(32,return_sequences=False))
  model.add(layers.Dense(num_class))
  return model
Input_dim= X_train_reshape
Classes = 1000

# # Creating the model on TPU
with tpu_strategy.scope():
  model = LSTM_Model(input_dim=Input_dim,num_class=Classes)
  model.compile(loss='mse',optimizer='adam',metrics=[tf.keras.metrics.RootMeanSquaredError()])
  from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
  es = EarlyStopping(monitor='val_loss',patience=10, min_delta=1e-3,mode='auto')
  model.summary()
  #tf.keras.optimizers.SGD(lr=config['train_params']['learn_rate'])


In [ ]:
# Training the LSTM model
LSTM_model_train = model.fit(X_train_reshape[:config['train_size']], # using the first 3 million rows.
                    y_train[:config['train_size']],
                    validation_split=0.1,
                    epochs=config['train_params']['epochs'],
                    batch_size=Batch_size,
                    callbacks=[es],
                    shuffle=False,
                    verbose = 1
                   )

In [ ]:
#Predicting on the test set using the LSTM model
LSTM_predictions = model.predict(X_test_reshape[:config['train_size']])
rmse = tf.keras.metrics.RootMeanSquaredError()
rmse.update_state(y_test[:config['train_size']], LSTM_predictions)
print(f'LSTM Model RMSE: {rmse.result().numpy()}')

In [ ]:
len(LSTM_predictions)

In [ ]:
type(LSTM_predictions)

In [ ]:
LSTM_predictions.shape

In [ ]:
LSTM_predictions

In [ ]:
y_test

In [ ]:
# MAE for LSTM Model
MAE= tf.keras.metrics.MeanAbsoluteError()
MAE.update_state(y_test, LSTM_predictions)
print(f'LSTM Model MAE: {MAE.result().numpy()}')

In [ ]:
# To check if there are no NaN values in LSTM_predictions and y_test
print(len(y_test))
print(len(LSTM_predictions))

print(np.isnan(y_test).any())
print(np.isnan(LSTM_predictions).any())

print(np.argwhere(np.isnan(y_test)))


In [ ]:
#Checking MAP error for LSTM model
score = []
for i in range(len(y_test)):
  score.append(abs(y_test[i]-LSTM_predictions[i]))

print("Average MAP error of LSTM model is: %f mmHG" %(np.mean(score)))

In [ ]:
#Visualizing LSTM model Train Loss vs RMSE in LSTM model
plt.title('Train loss against Root_Mean_Squared_error in LSTM model')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.plot(LSTM_model_train.history['loss'])
plt.plot(LSTM_model_train.history['root_mean_squared_error'])
plt.legend(['Loss', 'Root_Mean_Squared_error'])

In [ ]:
# Visualize predicted BP and the True BP for LSTM model
plt.title("===True BP values Vs Predicted BP values in LSTM Model===")
plt.xlabel('Number of samples taken')
plt.ylabel('BP values')
plt.plot(y_test[1000,:]) #only plotting 100 samples
plt.plot(LSTM_predictions[1000,:])
plt.legend(['True_BP', 'Predicted_BP'])